In [5]:
import pandapipes as pp
from pandapipes.pf.pipeflow_setup import get_fluid
import pandas as pd
import numpy as np

In [6]:
# =============================================================================
# CLASE ALMACENAMIENTO (una sola bomba reversible, sin flow controls)
# =============================================================================
class ThermoclineTwoLayer():
    def __init__(self, net, name, Circ_pump_id, j_supply, j_return,
                 volume_m3, t_hot_init, t_cold_init, v_hot_fraction_init,
                 UA_loss, UA_interface=0.0):
        self.net = net
        self.fluid = get_fluid(net)
        self.name = name
        self.Circ_pump_id = Circ_pump_id
        self.j_supply = j_supply      # nodo de ida donde se conecta el almacenamiento
        self.j_return = j_return      # nodo de retorno donde se conecta el almacenamiento
        self.V_tot = volume_m3
        self.T_hot = t_hot_init
        self.T_cold = t_cold_init
        self.v_hot_fraction = v_hot_fraction_init
        self.UA_loss = UA_loss
        self.UA_interface = UA_interface
        self.V_hot = volume_m3 * self.v_hot_fraction
        self.V_cold = volume_m3 - self.V_hot
        self.V_MIN = volume_m3 * 0.01
        # Variables operativas
        self.bypass = False
        self.direction = None
        self.mass_flow = 0.0
        self.mdot_entering = 0.0
        self.mdot_bypass = 0.0
        self.v_in = 0.0
        self.density_average_heat_capacity()

    def density_average_heat_capacity(self):
        T_min_K = 273.15 + 20
        T_max_K = 273.15 + 100
        T_samples = np.linspace(T_min_K, T_max_K, 100)
        rho_samples = np.array([self.fluid.get_density(T) for T in T_samples])
        cp_samples = np.array([self.fluid.get_heat_capacity(T) for T in T_samples])
        self.density = np.mean(rho_samples)
        self.heat_capacity = np.mean(cp_samples)

    # -------------------------------------------------------------------------
    # Configuración de la bomba (única)
    #   Carga   : toma de la ida (j_supply) e inyecta en el retorno a T_cold
    #   Descarga: toma del retorno (j_return) e inyecta en la ida a T_hot
    # La bomba nunca se apaga (in_service siempre True) y nunca tiene caudal 0
    # exacto: así el tamaño del pit no cambia y no hace falta parche.
    # -------------------------------------------------------------------------
    def _set_pump(self, mdot):
        n, p = self.net, self.Circ_pump_id
        mdot = max(abs(float(mdot)), 1e-4)
        if self.mass_flow >= 0:   # carga
            n.circ_pump_mass.at[p, "return_junction"] = self.j_supply
            n.circ_pump_mass.at[p, "flow_junction"] = self.j_return
            n.circ_pump_mass.at[p, "t_flow_k"] = self.T_cold
        else:                     # descarga
            n.circ_pump_mass.at[p, "return_junction"] = self.j_return
            n.circ_pump_mass.at[p, "flow_junction"] = self.j_supply
            n.circ_pump_mass.at[p, "t_flow_k"] = self.T_hot
        n.circ_pump_mass.at[p, "mdot_flow_kg_per_s"] = mdot
        n.circ_pump_mass.at[p, "in_service"] = True

    def evaluate_T_network_in(self):
        # Primera pasada: todo el caudal pedido entra al tanque (sin bypass)
        self._set_pump(self.mass_flow)
        pp.pipeflow(self.net, mode="bidirectional")
        return float(self.net.res_circ_pump_mass.at[self.Circ_pump_id, "t_from_k"])

    def evaluate_bypass(self, dt_s, T_net):
        bypass = False
        mdot_entering = 0.0
        mdot_bypass = 0.0
        v_in = 0.0
        if self.mass_flow > 0:  # Carga
            if T_net < self.T_hot:
                self.direction = "to_Tcold"
                mdot_entering = self.mass_flow
            else:
                self.direction = "to_Thot"
                v_available = max(0.0, self.V_cold - self.V_MIN)
                Dv_requested = (self.mass_flow * dt_s) / self.density
                v_in = min(Dv_requested, v_available)
                mdot_entering = (v_in * self.density) / dt_s if dt_s > 0 else 0.0
                mdot_bypass = self.mass_flow - mdot_entering
                if mdot_bypass > 1e-6 or v_available <= 0:
                    bypass = True
                    print(f"  ⚠ [{self.name}] Tanque saturado de calor: {Dv_requested - v_in:.2f} m3 no se pudieron cargar")
                    print(f"  ⚠ [{self.name}] Tanque saturado de calor: {mdot_bypass:.2f} kg/s no se pudieron cargar")
        elif self.mass_flow < 0:  # Descarga
            self.direction = "discharge"
            abs_mass_flow = abs(self.mass_flow)
            v_available = max(0.0, self.V_hot - self.V_MIN)
            Dv_requested = (abs_mass_flow * dt_s) / self.density
            v_in = min(Dv_requested, v_available)
            mdot_entering = (v_in * self.density) / dt_s if dt_s > 0 else 0.0
            mdot_bypass = abs_mass_flow - mdot_entering
            if mdot_bypass > 1e-6 or v_available <= 0:
                bypass = True
                print(f"  ⚠ [{self.name}] Tanque saturado de frio: {Dv_requested - v_in:.2f} m3 no se pudieron descargar")
                print(f"  ⚠ [{self.name}] Tanque saturado de frio: {mdot_bypass:.2f} kg/s no se pudieron descargar")
        else:
            self.direction = "static"
        self.bypass = bypass
        self.mdot_entering = mdot_entering
        self.mdot_bypass = mdot_bypass
        self.v_in = v_in

    def apply_control_settings(self):
        # El bypass es implícito: la bomba solo mueve lo que entra al tanque
        # (mdot_entering). El caudal de bypass ni se extrae ni se devuelve,
        # que es el mismo efecto neto que tenía el flow control de bypass.
        self._set_pump(self.mdot_entering)

    def Estimate_loss_ambient(self, T_amb):
        self.UA_hot_layer = self.UA_loss * (self.V_hot / self.V_tot)
        self.UA_cold_layer = self.UA_loss * (self.V_cold / self.V_tot)
        Q_loss_hot = self.UA_hot_layer * (self.T_hot - T_amb)
        Q_loss_cold = self.UA_cold_layer * (self.T_cold - T_amb)
        return Q_loss_hot, Q_loss_cold

    def V_dis(self, dt_s, T_amb, T_net):
        "Resolución con Euler implícito"
        rho, cp = self.density, self.heat_capacity
        b = (dt_s * self.UA_loss) / (rho * cp * self.V_tot)
        if self.mass_flow >= 0:
            if self.direction == 'to_Thot':
                self.V_cold -= self.v_in
                self.V_hot += self.v_in
                self.v_hot_fraction = self.V_hot / self.V_tot
                a_h = dt_s * self.mdot_entering / (rho * self.V_hot)
                self.T_hot = (self.T_hot + a_h * T_net + b * T_amb) / (1 + a_h + b)
                self.T_cold = (self.T_cold + b * T_amb) / (1 + b)
            elif self.direction == 'to_Tcold':
                a_c = dt_s * self.mdot_entering / (rho * self.V_cold)
                self.T_cold = (self.T_cold + a_c * T_net + b * T_amb) / (1 + a_c + b)
                self.T_hot = (self.T_hot + b * T_amb) / (1 + b)
            else:  # static
                self.T_hot = (self.T_hot + b * T_amb) / (1 + b)
                self.T_cold = (self.T_cold + b * T_amb) / (1 + b)
        else:  # descarga
            self.V_cold += self.v_in
            self.V_hot -= self.v_in
            self.v_hot_fraction = self.V_hot / self.V_tot
            a_c = dt_s * self.mdot_entering / (rho * self.V_cold)
            self.T_cold = (self.T_cold + a_c * T_net + b * T_amb) / (1 + a_c + b)
            self.T_hot = (self.T_hot + b * T_amb) / (1 + b)


In [7]:
# =============================================================================
# RED
# =============================================================================
net = pp.create_empty_network(fluid="water")

# Nudos de ida
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida")
j_cons_ida_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_1_Ida")
j_cons_ida_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_2_Ida")
j_cons_ida_3 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_3_Ida")

# Nudos de retorno
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret")
j_cons_ret_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_1_ret")
j_cons_ret_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_2_ret")
j_cons_ret_3 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_3_ret")

U_W = 0.35 / (np.pi * 0.15)

def make_pipe(fj, tj, length_km, name):
    return pp.create_pipe_from_parameters(
        net, from_junction=fj, to_junction=tj, length_km=length_km,
        inner_diameter_mm=150, u_w_per_m2k=U_W, text_k=273.2, name=name, k_mm=0.1)

# Planta - nodo 1
make_pipe(j_fuente_ida, j_nodo_1, 0.0589, "Tubo_Ida_Plant_Storage_ida")
make_pipe(j_nodo_1_ret, j_fuente_ret, 0.0589, "Tubo_RetornoPlant_Storage_ret")
# Nodo 1 - nodo 2
make_pipe(j_nodo_1, j_nodo_2, 0.0589, "Tubo_Ida_Storage_ida_nodo_1")
make_pipe(j_nodo_2_ret, j_nodo_1_ret, 0.0589, "Tubo_Retorno_Storage_ret_nodo_1")
# Consumidor 1
make_pipe(j_nodo_2, j_cons_ida_1, 0.0376, "Tubo_Ida_nodo_1_cons_1_ida")
make_pipe(j_cons_ret_1, j_nodo_2_ret, 0.0376, "Tubo_Retorno_nodo_1_cons_1_ret")
# Consumidor 2
make_pipe(j_nodo_2, j_cons_ida_2, 0.0302, "Tubo_Ida_nodo_1_cons_2_ida")
make_pipe(j_cons_ret_2, j_nodo_2_ret, 0.0302, "Tubo_Retorno_nodo_1_cons_2_ret")
# Consumidor 3
make_pipe(j_nodo_1, j_cons_ida_3, 0.0441, "Tubo_Ida_nodo_1_cons_3_ida")
make_pipe(j_cons_ret_3, j_nodo_2_ret, 0.0441, "Tubo_Retorno_nodo_1_cons_3_ret")

# Consumidores
HC_1 = pp.create_heat_consumer(net, from_junction=j_cons_ida_1, to_junction=j_cons_ret_1,
                               qext_w=150000, treturn_k=305, name="Consumidor_1")
HC_2 = pp.create_heat_consumer(net, from_junction=j_cons_ida_2, to_junction=j_cons_ret_2,
                               qext_w=150000, treturn_k=305, name="Consumidor_2")
HC_3 = pp.create_heat_consumer(net, from_junction=j_cons_ida_3, to_junction=j_cons_ret_3,
                               qext_w=150000, treturn_k=305, name="Consumidor_3")

# Planta (referencia de presión de toda la red)
Plant_1 = pp.create_circ_pump_const_pressure(
    net, flow_junction=j_fuente_ida, return_junction=j_fuente_ret,
    p_flow_bar=3, plift_bar=0.5, t_flow_k=360, name='Grid')

# Almacenamiento: UNA sola bomba reversible (carga por defecto)
Circ_pump_storage = pp.create_circ_pump_const_mass_flow(
    net, return_junction=j_nodo_1, flow_junction=j_nodo_1_ret,
    mdot_flow_kg_per_s=0.1, t_flow_k=313, p_flow_bar=None, type="t",
    name="Circ_pump_storage")

Storage = ThermoclineTwoLayer(
    net, "Storage_1", Circ_pump_id=Circ_pump_storage,
    j_supply=j_nodo_1, j_return=j_nodo_1_ret,
    volume_m3=10, t_hot_init=354, t_cold_init=313, v_hot_fraction_init=2/10,
    UA_loss=15, UA_interface=0.0)

# =============================================================================
# DATOS
# =============================================================================
Data = pd.read_excel("C:\\Users\\sserranose\\OneDrive - INSA Lyon\\Bureau\\Code\\Panda_pipes\\Storage\\Profiles_For_Energy_Storage.xlsx")
Q_consumer_1 = Data.iloc[:, 2].to_numpy()
Q_consumer_2 = Data.iloc[:, 3].to_numpy()
Q_consumer_3 = Data.iloc[:, 4].to_numpy()
Débit_Storage = Data.iloc[:, 5].to_numpy() * -1

keys = ["hora", "Q_plant_PD", "Q_loss_storage_PD", "Q_Storage_PD", "T_hot_storage_PD",
        "T_cold_storage_PD", 'V_hot_PD', 'V_cold_PD', 'bypass', 'direction', 'mdot_entering',
        'mdot_bypass', 'm_dot_bypass_panda', 'volumen_bypass', 'Tnet_1', "T_net_2"]
results = {key: [] for key in keys}

dt_seconds = 3600
T_ambient = 273.2


def log_state(i, etiqueta):
    with open("resultados.txt", "a", encoding="utf-8") as f:
        f.write(f"iteration {i} - {etiqueta}\n\n")
        f.write("circ_pump_mass\n" + net.circ_pump_mass.to_string() + "\n\n")
        f.write("res_circ_pump_mass\n" + net.res_circ_pump_mass.to_string() + "\n\n")
        f.write("res_junction\n" + net.res_junction.to_string() + "\n\n")

In [8]:

# =============================================================================
# BUCLE PRINCIPAL
# =============================================================================
for i in range(len(Débit_Storage)):
    print(i + 1)

    # 1. Consumidores
    for HC, Q in ((HC_1, Q_consumer_1), (HC_2, Q_consumer_2), (HC_3, Q_consumer_3)):
        net.heat_consumer.at[HC, "in_service"] = bool(Q[i] > 0)
        if Q[i] > 0:
            net.heat_consumer.at[HC, "qext_w"] = Q[i]

    # 2. Caudal del almacenamiento (elimina el -0.0)
    mdot_target = float(Débit_Storage[i])
    if abs(mdot_target) < 1e-6:
        mdot_target = 0.0
    Storage.mass_flow = mdot_target

    # 3. Primera pasada: temperatura de entrada al tanque sin bypass
    try:
        T_network_preliminar = Storage.evaluate_T_network_in()
    except Exception:
        print("FALLA en pasada 1, i =", i, "| mass_flow =", Storage.mass_flow,
              "| Q =", Q_consumer_1[i], Q_consumer_2[i], Q_consumer_3[i])
        print(net.circ_pump_mass[["in_service", "return_junction", "flow_junction",
                                  "mdot_flow_kg_per_s", "t_flow_k"]])
        raise
    log_state(i, "antes")

    # 4. Bypass y segunda pasada con el caudal que realmente entra
    Storage.evaluate_bypass(dt_s=dt_seconds, T_net=T_network_preliminar)
    Storage.apply_control_settings()
    try:
        pp.pipeflow(net, mode="bidirectional")
    except Exception:
        print("FALLA en pasada 2, i =", i, "| mass_flow =", Storage.mass_flow,
              "| mdot_entering =", Storage.mdot_entering, "| mdot_bypass =", Storage.mdot_bypass)
        print(net.circ_pump_mass[["in_service", "return_junction", "flow_junction",
                                  "mdot_flow_kg_per_s", "t_flow_k"]])
        raise
    log_state(i, "despues")

    T_network_definitiva = float(net.res_circ_pump_mass.at[Storage.Circ_pump_id, "t_from_k"])

    # 5. Pérdidas y evolución del tanque
    Q_loss_hot, Q_loss_cold = Storage.Estimate_loss_ambient(T_amb=T_ambient)
    Storage.V_dis(dt_s=dt_seconds, T_amb=T_ambient, T_net=T_network_definitiva)

    print(f"  ⚠ [{Storage.name}] Temperatura antes: {T_network_preliminar:.2f} K, después: {T_network_definitiva:.2f} K")

    # 6. Resultados
    results["hora"].append(i + 1)
    results["Q_plant_PD"].append(float(net.res_circ_pump_pressure.at[Plant_1, "qext_w"]))
    results["Q_loss_storage_PD"].append(float(Q_loss_hot + Q_loss_cold))
    results["Q_Storage_PD"].append(float(net.res_circ_pump_mass.at[Storage.Circ_pump_id, "qext_w"]))
    results['V_hot_PD'].append(Storage.V_hot)
    results['V_cold_PD'].append(Storage.V_cold)
    results["T_hot_storage_PD"].append(float(Storage.T_hot))
    results["T_cold_storage_PD"].append(float(Storage.T_cold))
    results['bypass'].append(Storage.bypass)
    results['direction'].append(Storage.direction)
    results['mdot_entering'].append(Storage.mdot_entering)
    results['mdot_bypass'].append(Storage.mdot_bypass)
    results['m_dot_bypass_panda'].append(Storage.mdot_bypass)
    results['volumen_bypass'].append(
        Storage.mdot_bypass / Storage.fluid.get_density(T_network_definitiva) * dt_seconds)
    results['Tnet_1'].append(float(T_network_preliminar))
    results['T_net_2'].append(T_network_definitiva)

df = pd.DataFrame(results)
df.to_excel("output_real_thermocline_7_10.xlsx", index=False, sheet_name="Sheet1")

1
  ⚠ [Storage_1] Temperatura antes: 359.79 K, después: 359.79 K
2
  ⚠ [Storage_1] Temperatura antes: 359.85 K, después: 359.85 K
3
  ⚠ [Storage_1] Temperatura antes: 304.77 K, después: 304.77 K
4
  ⚠ [Storage_1] Temperatura antes: 304.73 K, después: 304.73 K
5
  ⚠ [Storage_1] Temperatura antes: 359.82 K, después: 359.82 K
6
  ⚠ [Storage_1] Tanque saturado de calor: 0.10 m3 no se pudieron cargar
  ⚠ [Storage_1] Tanque saturado de calor: 0.03 kg/s no se pudieron cargar
  ⚠ [Storage_1] Temperatura antes: 359.80 K, después: 359.79 K
7
  ⚠ [Storage_1] Tanque saturado de calor: 2.75 m3 no se pudieron cargar
  ⚠ [Storage_1] Tanque saturado de calor: 0.75 kg/s no se pudieron cargar
  ⚠ [Storage_1] Temperatura antes: 359.80 K, después: 359.68 K
8
  ⚠ [Storage_1] Temperatura antes: 304.72 K, después: 304.72 K
9
  ⚠ [Storage_1] Temperatura antes: 304.72 K, después: 304.72 K
10
  ⚠ [Storage_1] Temperatura antes: 359.76 K, después: 359.76 K
11
  ⚠ [Storage_1] Temperatura antes: 304.73 K, después: 